# Coffee spend by age and gender: Type II sums of squares

**Two-Way Analysis of Variance** · a worked example from the StatNex teaching guide

**The story.** A cafe looked at 13 regular customers' monthly coffee spending, together with their age group and gender, which are both categories rather than numbers.

**The question.** Which matters more for coffee spending: age group or gender?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `09_multivariate/quantification_i.csv` (13 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Coffee spend ($/month),Age group,Gender
1.0,18.0,Teens,Female
2.0,25.0,Teens,Male
3.0,22.0,Teens,Female
4.0,45.0,20s,Male
5.0,60.0,20s,Female
6.0,52.0,20s,Male
7.0,75.0,20s,Female
8.0,38.0,20s,Male
9.0,70.0,30s,Male
10.0,55.0,30s,Female
11.0,82.0,30s,Male
12.0,40.0,40s,Male
13.0,48.0,40s,Female

## Libraries

In [ ]:
from itertools import combinations

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.anova import anova_lm

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per observation: the response and the two factors. Rows with a blank are
# dropped. Levels keep the order they first appear in; every combination of levels
# needs data, and some cell needs at least two observations.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "Coffee spend ($/month)"
column_factor = "Gender"
row_factor = "Age group"
frame = pd.DataFrame({"y": pd.to_numeric(data[response], errors="coerce"),
                      "A": data[column_factor], "B": data[row_factor]}).dropna()
column_levels = list(frame["A"].unique())
row_levels = list(frame["B"].unique())
frame["A"] = pd.Categorical(frame["A"], categories=column_levels)
frame["B"] = pd.Categorical(frame["B"], categories=row_levels)

## Descriptive statistics

In [ ]:
# Each level of the column factor (A), of the row factor (B), then each cell.
def describe(by):
    table = frame.groupby(by, observed=True)["y"].agg(n="count", mean="mean", SD="std")
    table.index = [" * ".join(map(str, np.atleast_1d(key))) for key in table.index]
    table["SE"] = table["SD"] / np.sqrt(table["n"])
    return table.assign(**{"mean - SD": table["mean"] - table["SD"],
                           "mean + SD": table["mean"] + table["SD"],
                           "mean - SE": table["mean"] - table["SE"],
                           "mean + SE": table["mean"] + table["SE"]})[
        ["n", "mean", "SD", "mean - SD", "mean + SD", "SE", "mean - SE", "mean + SE"]]


descriptive = pd.concat([describe("A"), describe("B"), describe(["A", "B"])],
                        keys=["A", "B", "A * B"])
print(descriptive.to_string())

## Test for homogeneity of variance

In [ ]:
# Levene's original form across the cells, on deviations from each cell's mean
# (center="mean"; SciPy's default is the median).
cells = [group["y"].to_numpy() for _, group in frame.groupby(["A", "B"], observed=True)]
levene = stats.levene(*cells, center="mean")
homogeneity = pd.DataFrame([[levene.statistic, len(cells) - 1, len(frame) - len(cells),
                             levene.pvalue]], columns=["F", "df1", "df2", "P"])
print()
print(homogeneity.to_string(index=False))

## Analysis of variance table

In [ ]:
# The effects use sum-to-zero coding (C(..., Sum)). With equal cell sizes the three
# types of sums of squares coincide with the textbook decomposition; with unequal
# cells they differ in what each main effect is adjusted for. The interaction is
# entered last in every type, so its line is the same in all three.
fit = smf.ols("y ~ C(A, Sum) * C(B, Sum)", data=frame).fit()
# Type II: each main effect adjusted for the other main effect, but not for the
# interaction.
table = anova_lm(fit, typ=2)
ss_error, df_error = table.loc["Residual", "sum_sq"], table.loc["Residual", "df"]
ms_error = ss_error / df_error
anova = pd.DataFrame({"SS": table["sum_sq"].to_numpy(), "df": table["df"].to_numpy()},
                     index=["A (column)", "B (row)", "A * B", "Error"])
anova["MS"] = anova["SS"] / anova["df"]
anova["F"] = anova["MS"] / ms_error
anova["P"] = stats.f.sf(anova["F"], anova["df"], df_error)
anova.loc["Error", ["F", "P"]] = np.nan
anova.loc["Total"] = [((frame["y"] - frame["y"].mean()) ** 2).sum(), len(frame) - 1,
                      np.nan, np.nan, np.nan]
print()
print(anova)

columns = ["level 1", "level 2", "mean 1", "mean 2", "difference", "SE", "statistic", "P"]

## Least-squares means

In [ ]:
# The mean each level would have if every cell weighed the same: the average of its
# cell means, with SE = sqrt(MS error x sum(1 / n of each cell)) / number of cells.
# With equal cell sizes this is the plain mean.
confidence = 95 / 100
critical = stats.t.ppf(0.5 + confidence / 2, df_error)
cell_mean = frame.pivot_table("y", "A", "B", aggfunc="mean", observed=True)
cell_n = frame.pivot_table("y", "A", "B", aggfunc="count", observed=True)
rows = []
for c in column_levels:
    se = np.sqrt(ms_error * (1 / cell_n.loc[c]).sum()) / len(row_levels)
    rows.append([c, cell_n.loc[c].sum(), cell_mean.loc[c].mean(), se])
for r in row_levels:
    se = np.sqrt(ms_error * (1 / cell_n[r]).sum()) / len(column_levels)
    rows.append([r, cell_n[r].sum(), cell_mean[r].mean(), se])
for c in column_levels:
    for r in row_levels:
        rows.append([f"{c} * {r}", cell_n.loc[c, r], cell_mean.loc[c, r],
                     np.sqrt(ms_error / cell_n.loc[c, r])])
lsmeans = pd.DataFrame(rows, columns=["level", "n", "LS mean", "SE"])
lsmeans["lower"] = lsmeans["LS mean"] - critical * lsmeans["SE"]
lsmeans["upper"] = lsmeans["LS mean"] + critical * lsmeans["SE"]
print()
print(lsmeans.to_string(index=False))

## What the result says

Two-way analysis of variance (ANOVA) tests whether mean coffee spend differs by gender and by age group. The 13 customers are spread unevenly over the groups, so the result can depend on how each effect is tested. Type II sums of squares test each factor after the other factor, so neither gets credit for what the other explains. The order of the factors no longer matters.

Gender, allowing for age group, has a sum of squares of 61.6 and F = 1.06, with P = 0.35. The P value is the chance of an F this large if there were no real difference, so no gender difference is shown. Age group, allowing for gender, gives F = 20.8 and P = 0.003: a clear difference. The interaction, a different age pattern for women and men, has F = 5.10 and P = 0.056, and is not shown.

So the answer holds: age group matters far more than gender.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.